# run\_d002 Short-Form Random 50 Clips

This notebook isolates the `run_d002` diffusion checkpoint on its own, without long-form recursion.

Goal:
- hear what `run_d002` sounds like on random songs from `Downloads/`
- judge short-form realism and style shift directly
- separate checkpoint quality from Lab 4 long-form artifact compounding

Default behavior:
- uses `run_d002/checkpoints/best.pt`
- generates 50 random 3-second clips
- saves source excerpts, generated audio, and a `manifest.csv`

If you want the documented subjective checkpoint instead, switch `CHECKPOINT_NAME` to `epoch_006.pt` in the config cell.


In [ ]:
from pathlib import Path
import importlib
import json
import sys

import pandas as pd

def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'dggr').exists() and (path / 'lab 3').exists() and (path / 'lab 3.1').exists():
            return path
    raise RuntimeError('Could not resolve repo root from current working directory.')

REPO = find_repo_root(Path.cwd().resolve())
SCRIPTS = REPO / 'lab 3.1' / 'scripts'
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

import diffusion_downloads_batch as ddb
importlib.reload(ddb)
REPO

In [ ]:
RUN_DIR = REPO / 'saves2' / 'lab3_diffusion' / 'run_d002'
CHECKPOINT_NAME = 'best.pt'   # change to 'epoch_006.pt' if you want the subjective checkpoint
CHECKPOINT_PATH = RUN_DIR / 'checkpoints' / CHECKPOINT_NAME

cfg = ddb.DiffusionDownloadsBatchConfig(
    downloads_dir=Path.home() / 'Downloads',
    output_root=REPO / 'lab 3.1' / 'outputs' / 'run_d002_shortform_random50',
    run_dir=RUN_DIR,
    checkpoint_path=CHECKPOINT_PATH,
    cache_dir=REPO / 'saves2' / 'lab3_diffusion' / 'run_d001' / 'cache',
    n_clips=50,
    clip_seconds=3.0,
    n_frames=256,
    ddim_steps=50,
    guidance_scale=2.0,
    t_start=320,
    style_strength=0.90,
    device='auto',
    seed=328,
)

RUN_ALL = True

ctx = ddb.resolve_inference_context(cfg)
print('Resolved run dir:      ', ctx['run_dir'])
print('Resolved checkpoint:   ', ctx['checkpoint_path'])
print('Resolved cache dir:    ', ctx['cache_dir'])
print('Planned output root:   ', cfg.output_root / cfg.tag)

In [ ]:
jobs = ddb.plan_jobs(cfg)
preview = pd.DataFrame(jobs)
display(preview.head(12))
print('Total planned clips:', len(preview))
print('Target counts:')
display(preview['target_genre'].value_counts().sort_index())

In [ ]:
summary = None
if RUN_ALL:
    summary = ddb.run_batch_inference(cfg)
    print(json.dumps(summary, indent=2, default=str))
else:
    print('Set RUN_ALL = True to generate the 50 random run_d002 clips.')

In [ ]:
summary_path = cfg.output_root / cfg.tag / 'summary.json'
manifest_path = cfg.output_root / cfg.tag / 'manifest.csv'
if summary_path.exists():
    print(summary_path)
    print(manifest_path)
    display(pd.read_csv(manifest_path).head(20))
else:
    print('No outputs yet for this tag.')